# Get genes with halo formation activity from Halo Assay data

In [119]:
import polars as pl

In [120]:
# data from Raphael, converted to csv from xlsx
df = pl.scan_csv("./data/Source_data_v1.csv").collect()
df.columns = ['construct',	'BHET conc', 'timepoint', 'gene', 'colony intensity adjusted', 'quad intensity adjusted', 'colony intensity p-val', 'quad intensity p-val']

In [121]:
# remove first 12 rows, which are headers
df = df[12:]

In [122]:
df = df.with_columns(
    pl.col('timepoint').cast(pl.Float64),
    pl.col('colony intensity adjusted').cast(pl.Float64),
    pl.col('quad intensity adjusted').cast(pl.Float64),
    pl.col('colony intensity p-val').cast(pl.Float64),
    pl.col('quad intensity p-val').cast(pl.Float64)
)

In [123]:
# take all genes with a significant colony intensity p-value (< 0.05) at any timepoint. We can afford to be liberal here,
# since we're just ranking the biosamples, and we don't want to miss any potentially interesting genes
# Note that we don't filter on quad intensity p-value, since that appears to include a lot of junk? Empty well has a < 0.05 quad p-val
active = df.filter(
    (pl.col('colony intensity p-val') < 0.05)
).select(
    ['construct', 'gene', 'timepoint', 'colony intensity adjusted', 'quad intensity adjusted', 'colony intensity p-val', 'quad intensity p-val']
)

In [128]:
prefixes = ["SRR", "ERR", "DRR"]
active = active.filter(
    pl.any_horizontal([pl.col("gene").str.starts_with(p) for p in prefixes]))
# remove suffixes after "_" 
active = active.with_columns(
    pl.col("gene").str.split("_").list.first().alias("gene")
)
# add a column "count" that counts the number of times each gene appears in the dataframe
active = active.with_columns(
    pl.len().over("gene").alias("count")
)
# divide count by 4 because of replicas
active = active.with_columns(
    (pl.col("count") / 4).alias("count")
)

# remove duplicates of gene, keeping the one with the highest count
active = active.sort("count", descending=True).unique(subset=["gene"], keep="first")

In [131]:
active.write_parquet("./data/active_genes.parquet")

# Result
`active_genes.parquet` now contains a list of accessions that contain a gene which expressed activity in the Halo Assay. These will be used to rank-order 90%pid clusters for assaying